# Mini-Batch K-Means: `partial_fit`

Train with **incremental** minibatches (online / streaming) instead of a single full pass via `fit`.

**Reference:** David Sculley, *Web-Scale K-Means Clustering*, WWW 2010. DOI [10.1145/1772690.1772862](https://doi.org/10.1145/1772690.1772862).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT.resolve()))

import torch
from minibatch_kmeans import MiniBatchKMeans

## Synthetic 2D data

Three Gaussian blobs (easy to visualize).

In [ ]:
g = torch.Generator().manual_seed(0)
blob_centers = torch.tensor([[0.0, 0.0], [5.0, 0.0], [2.5, 4.0]], dtype=torch.float32)
k = blob_centers.shape[0]
n_per = 200
parts = [
    blob_centers[i] + 0.35 * torch.randn(n_per, 2, generator=g)
    for i in range(k)
]
X = torch.cat(parts, dim=0)
X.shape, k

## Stream minibatches with `partial_fit`

Each batch must have at least `n_clusters` points (here `k=3`).

In [ ]:
batch_size = 64
epochs = 4
n_samples = X.shape[0]
perm_rng = torch.Generator().manual_seed(1)
km = MiniBatchKMeans(
    n_clusters=k,
    random_state=torch.Generator().manual_seed(42),
    dtype=torch.float32,
)

for _ in range(epochs):
    perm = torch.randperm(n_samples, generator=perm_rng)
    for start in range(0, n_samples, batch_size):
        idx = perm[start : start + batch_size]
        if idx.numel() < k:
            continue
        km.partial_fit(X[idx])

km.cluster_centers.shape

## Optional: compare to one-shot `fit`

Same hyperparameters on the full tensor (different optimization path, comparable role).

In [ ]:
km_batch = MiniBatchKMeans(
    n_clusters=k,
    random_state=torch.Generator().manual_seed(42),
    dtype=torch.float32,
)
km_batch.fit(X, batch_size=batch_size, max_iter=2)
km_batch.cluster_centers.shape

## Plot (requires matplotlib)

`pip install matplotlib` if needed.

In [ ]:
import matplotlib.pyplot as plt

labels = km.predict(X)
centers = km.cluster_centers.detach().cpu().numpy()
xy = X.detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(xy[:, 0], xy[:, 1], c=labels.cpu().numpy(), s=10, alpha=0.65, cmap="tab10")
ax.scatter(centers[:, 0], centers[:, 1], c="red", marker="x", s=120, linewidths=2, label="centers")
ax.set_title("partial_fit streaming model")
ax.legend()
ax.set_aspect("equal", adjustable="box")
plt.colorbar(sc, ax=ax, label="cluster")
plt.tight_layout()
plt.show()